IMPORTACIONES

In [1]:
#Importar librerías y registrar versiones
import sys, time, json, pickle, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Mismas rutas que los notebooks 08 y 09
from Configuracion import MODELOS_ML

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python       :", sys.version.split()[0])
print("scikit-learn :", sklearn.__version__)
print("pandas       :", pd.__version__)
print("numpy        :", np.__version__)

ADVERTENCIA: La ruta de Hadoop 'C:\Users\galle\Desktop\IA Datos\hadoop' no existe. Revisa Configuracion.py
Python       : 3.14.4
scikit-learn : 1.9.1
pandas       : 3.0.6
numpy        : 2.5.3


CARGAR SPLIT

In [ ]:
#Cargar  el split
RUTA_PKL = MODELOS_ML / "datos_split_guia8.pkl"

with open(RUTA_PKL, "rb") as f:        
    datos = pickle.load(f)

X_train, y_train = datos["X_train"], datos["y_train"]
X_val,   y_val   = datos["X_val"],   datos["y_val"]     
X_test,  y_test  = datos["X_test"],  datos["y_test"]    

COLS_NUMERICAS   = datos["COLS_NUMERICAS"]
COLS_CATEGORICAS = datos["COLS_CATEGORICAS"]
TARGET           = datos["TARGET"]
features = COLS_NUMERICAS + COLS_CATEGORICAS

print(f"Entrenamiento: {len(X_train):,} | Validación (sin usar): {len(X_val):,} | Prueba (aislada): {len(X_test):,}")
print("Variables predictoras:", len(features), "| Objetivo:", TARGET)

Entrenamiento: 396,267 | Validación (sin usar): 84,915 | Prueba (aislada): 84,915
Variables predictoras: 21 | Objetivo: es_ataque


In [3]:
# Verificaciones de integridad del split 
assert X_train.index.intersection(X_test.index).empty, "Hay filas de train en test"
assert X_train.index.intersection(X_val.index).empty,  "Hay filas de train en val"
assert list(X_train.columns) == features, "Las columnas no coinciden con las de la parte 8"

print("% ataque en train:", round(y_train.mean() * 100, 2))
print("% ataque en test :", round(y_test.mean() * 100, 2), "(solo para confirmar la estratificación)")
print("Nulos en train   :", int(X_train.isnull().sum().sum()))

% ataque en train: 19.73
% ataque en test : 19.73 (solo para confirmar la estratificación)
Nulos en train   : 0


RECUPERAR REFERENCIA

In [4]:
#Tabla de referencia (valores tomados de la salida del notebook 09)
referencia = pd.DataFrame([
    {"Modelo": "Baseline (Dummy most_frequent)", "Accuracy": 0.8027, "Precision": 0.0000, "Recall": 0.0000, "F1": 0.0000, "ROC-AUC": 0.5000},
    {"Modelo": "Regresión Logística (G8/G9)",     "Accuracy": 0.8510, "Precision": 0.5732, "Recall": 0.9571, "F1": 0.7170, "ROC-AUC": 0.9573},
    {"Modelo": "Candidato Guía 9: Random Forest", "Accuracy": 0.9971, "Precision": 0.9886, "Recall": 0.9967, "F1": 0.9926, "ROC-AUC": 0.9998},
    {"Modelo": "Optimizado Guía 10",              "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan, "F1": np.nan, "ROC-AUC": np.nan},
])
referencia["Observación"] = ["Referencia mínima", "Modelo base de la Guía 8", "Configuración por defecto, n_estimators=100", "Pendiente (Parte I)"]
print(referencia.to_string(index=False))

                         Modelo  Accuracy  Precision  Recall     F1  ROC-AUC                                 Observación
 Baseline (Dummy most_frequent)    0.8027     0.0000  0.0000 0.0000   0.5000                           Referencia mínima
    Regresión Logística (G8/G9)    0.8510     0.5732  0.9571 0.7170   0.9573                    Modelo base de la Guía 8
Candidato Guía 9: Random Forest    0.9971     0.9886  0.9967 0.9926   0.9998 Configuración por defecto, n_estimators=100
             Optimizado Guía 10       NaN        NaN     NaN    NaN      NaN                         Pendiente (Parte I)


In [5]:
# Referencia de validación cruzada
ruta_cv9 = MODELOS_ML / "validacion_cruzada_guia09.csv"
if ruta_cv9.exists():
    cv9 = pd.read_csv(ruta_cv9, encoding="utf-8-sig")
    print(cv9.to_string(index=False))
else:
    print("No se encontró", ruta_cv9, "→ referencia CV G9: Random Forest F1 = 0.9930 ± 0.0007")

                      Modelo  F1_CV_Media  F1_CV_DesvStd  Recall_CV_Media  Estable (Desv < 0.02)
          Dummy (Línea base)       0.0000         0.0000           0.0000                   True
Regresión Logística (modelo)       0.7146         0.0026           0.9570                   True
   Random Forest (modelo_rf)       0.9930         0.0007           0.9961                   True


In [6]:
# Cargar el candidato para ver su configuración exacta.
ruta_modelo9 = MODELOS_ML / "modelo_candidato_guia09.joblib"
ruta_ficha9  = MODELOS_ML / "ficha_modelo_guia09.json"

if ruta_modelo9.exists():
    sha_actual = hashlib.sha256(ruta_modelo9.read_bytes()).hexdigest()[:16]
    if ruta_ficha9.exists():
        sha_esperado = json.loads(ruta_ficha9.read_text(encoding="utf-8"))["sha256"]
        assert sha_actual == sha_esperado, f"Huella distinta: {sha_actual} vs {sha_esperado}"
        print("Huella SHA-256 verificada:", sha_actual)
    candidato_g9 = joblib.load(ruta_modelo9)
    p = candidato_g9.named_steps["clf"].get_params()
    print({k: p[k] for k in ["n_estimators", "max_depth", "min_samples_leaf", "max_features", "class_weight", "random_state"]})
else:
    print("Candidato G9 no encontrado; se usarán los valores por defecto de RandomForestClassifier.")

Huella SHA-256 verificada: d85c38d7b87c2c21
{'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced', 'random_state': 42}


PIPELINE REPDROCUCIBLE

In [7]:
# Pipeline con ColumnTransformer (hay variables categóricas: protocol y categoria_duracion)
preprocesador = ColumnTransformer([
    ("num", StandardScaler(), COLS_NUMERICAS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), COLS_CATEGORICAS),
])

pipeline = Pipeline([
    ("prep", preprocesador),
    ("clf", RandomForestClassifier(
        class_weight="balanced",     
        random_state=RANDOM_STATE,
        n_jobs=1                     
    )),
])
print(pipeline)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['flow_duration',
                                                   'total_fwd_packets',
                                                   'total_backward_packets',
                                                   'total_length_of_fwd_packets',
                                                   'total_length_of_bwd_packets',
                                                   'fwd_packet_length_mean',
                                                   'bwd_packet_length_mean',
                                                   'packet_length_mean',
                                                   'packet_length_std',
                                                   'flow_bytes_s',
                                                   'flow_packets_s',
                                                   'flow_iat_mean',
                

espacio de búsqueda
**Justificación de rangos:**

| Hiperparámetro | Valores | Por qué |
|---|---|---|
| `n_estimators` | 100, 200 | 100 es el valor de la Guía 9; 200 prueba si más árboles reduce la varianza. |
| `max_depth` | 10, 20, None | El candidato (`None`) llegó a F1 train 0.9956 vs test 0.9926. Se exploran árboles más superficiales como regularización. |
| `min_samples_leaf` | 1, 3, 5 | 1 es el valor por defecto; 3 y 5 suavizan hojas muy específicas. |
| `max_features` | "sqrt", 0.5 | "sqrt" es el valor por defecto en clasificación; 0.5 evalúa más variables por división. |


In [8]:
#Grid
param_grid = {
    "clf__n_estimators":     [100, 200],
    "clf__max_depth":        [10, 20, None],
    "clf__min_samples_leaf": [1, 3, 5],
    "clf__max_features":     ["sqrt", 0.5],
}

combinaciones = list(ParameterGrid(param_grid))
N_SPLITS = 4
print("Combinaciones            :", len(combinaciones))
print("Entrenamientos (x folds) :", len(combinaciones) * N_SPLITS, "+ 1 refit final")

config_g9 = {"clf__n_estimators": 100, "clf__max_depth": None, "clf__min_samples_leaf": 1, "clf__max_features": "sqrt"}
assert config_g9 in combinaciones, "El grid debe contener la configuración de la Guía 9"
print("La configuración de la Guía 9 está dentro del grid ✔")

Combinaciones            : 36
Entrenamientos (x folds) : 144 + 1 refit final
La configuración de la Guía 9 está dentro del grid ✔


CONFIGURAR VALIDACION CRUZADA

In [9]:
# Folds estratificados dentro del entrenamiento
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (idx_tr, idx_val) in enumerate(cv.split(X_train, y_train), 1):
    # control de fuga: ningún registro puede estar a la vez en train y validación del fold
    assert np.intersect1d(X_train.index[idx_tr], X_train.index[idx_val]).size == 0
    print(f"Fold {fold}: train={len(idx_tr):,} | validación={len(idx_val):,} | "
            f"% ataque train={y_train.iloc[idx_tr].mean()*100:.2f} | % ataque val={y_train.iloc[idx_val].mean()*100:.2f}")
print("Sin solapamiento train/validación en ningún fold ")

Fold 1: train=297,200 | validación=99,067 | % ataque train=19.73 | % ataque val=19.73
Fold 2: train=297,200 | validación=99,067 | % ataque train=19.73 | % ataque val=19.73
Fold 3: train=297,200 | validación=99,067 | % ataque train=19.73 | % ataque val=19.73
Fold 4: train=297,201 | validación=99,066 | % ataque train=19.73 | % ataque val=19.73
Sin solapamiento train/validación en ningún fold 


CONFIGURACION Y EJECUCUION DE GRIDSEARCHCV

In [10]:
# Configuración de GridSearchCV
MODO_PRUEBA = False      
N_JOBS      = -1         

scoring = {"f1": "f1", "recall": "recall", "precision": "precision"}

busqueda = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit="f1",               
    cv=cv,
    n_jobs=N_JOBS,
    return_train_score=True,   
    error_score="raise",
    verbose=2,
)

if MODO_PRUEBA:
    X_busq, _, y_busq, _ = train_test_split(
        X_train, y_train, train_size=20_000, stratify=y_train, random_state=RANDOM_STATE
    )
else:
    X_busq, y_busq = X_train, y_train

print("Filas usadas en la búsqueda:", f"{len(X_busq):,}", "| MODO_PRUEBA =", MODO_PRUEBA)

Filas usadas en la búsqueda: 396,267 | MODO_PRUEBA = False


In [11]:
#Ejecutar la búsqueda sobre entrenamiento
inicio = time.perf_counter()
busqueda.fit(X_busq, y_busq)
tiempo_total = time.perf_counter() - inicio

i = busqueda.best_index_
res = busqueda.cv_results_
print(f"\nTiempo total de búsqueda: {tiempo_total/60:.1f} min ({tiempo_total:.0f} s)")
print("Mejores parámetros      :", busqueda.best_params_)
print(f"F1 CV     : {res['mean_test_f1'][i]:.4f} ± {res['std_test_f1'][i]:.4f}")
print(f"Recall CV : {res['mean_test_recall'][i]:.4f} ± {res['std_test_recall'][i]:.4f}")
print(f"Precision : {res['mean_test_precision'][i]:.4f} ± {res['std_test_precision'][i]:.4f}")

Fitting 4 folds for each of 36 candidates, totalling 144 fits
[CV] END clf__max_depth=10, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__n_estimators=100; total time= 1.4min
[CV] END clf__max_depth=10, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__n_estimators=100; total time= 1.5min
[CV] END clf__max_depth=10, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__n_estimators=100; total time= 1.5min
[CV] END clf__max_depth=10, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__n_estimators=100; total time= 1.5min
[CV] END clf__max_depth=10, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__n_estimators=200; total time= 3.0min
[CV] END clf__max_depth=10, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__n_estimators=200; total time= 3.1min
[CV] END clf__max_depth=10, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__n_estimators=200; total time= 3.1min
[CV] END clf__max_depth=10, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__n_estimators=200; total

In [12]:
# Punto de control: guardar el registro completo de la búsqueda (cv_results_) 
resultados = pd.DataFrame(busqueda.cv_results_)
sufijo = "_PRUEBA" if MODO_PRUEBA else ""
ruta_cv_results = MODELOS_ML / f"gridsearch_resultados_guia10{sufijo}.csv"
resultados.to_csv(ruta_cv_results, index=False, encoding="utf-8")
print("cv_results_ guardado en:", ruta_cv_results.resolve())

# Vista rápida: top 5 por F1 de validación
vista = resultados[["rank_test_f1", "mean_test_f1", "std_test_f1", "mean_test_recall",
                    "mean_train_f1", "mean_fit_time", "params"]].sort_values("rank_test_f1").head(5)
print(vista.round(4).to_string(index=False))

cv_results_ guardado en: /home/vctor/Proyecto_GDIABD_Cybersecurity_Data_Analytics/Datos/Modelos/gridsearch_resultados_guia10.csv
 rank_test_f1  mean_test_f1  std_test_f1  mean_test_recall  mean_train_f1  mean_fit_time                                                                                                      params
            1        0.9939       0.0005            0.9962         0.9975       184.2595    {'clf__max_depth': None, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 100}
            2        0.9938       0.0003            0.9964         0.9974       382.4958    {'clf__max_depth': None, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 200}
            3        0.9933       0.0005            0.9972         0.9955       190.7889    {'clf__max_depth': None, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 3, 'clf__n_estimators': 100}
            4        0.9932       0.0005            0.9972         0.9955       369